# CS4106 Session 9 — Build an Open-Source RAG App
Run each cell with Shift+Enter. **Never type your Groq key into code.**

## Segment 1 — Set up & connect to Groq

In [ ]:
# Cell 1 — install everything (run once; ~1 min)
!pip install -q streamlit groq sentence-transformers faiss-cpu PyPDF2 python-docx

In [ ]:
# Cell 2 — enter your Groq key the SAFE way (stays hidden)
import os, getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your Groq API key (it will stay hidden): ")

In [ ]:
# Cell 3 — test that Groq works
from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])
resp = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "In one sentence, what is RAG?"}],
)
print(resp.choices[0].message.content)

## Segment 2 — Embeddings: text → vectors

In [ ]:
# Cell 4 — load the free embedding model and feel how it works
import numpy as np
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

v = embedder.encode(["I love this course", "This class is great", "The weather is cold"])
def cos(a, b): return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
print("similar meanings :", round(cos(v[0], v[1]), 3))   # expect HIGH (~0.6+)
print("different meanings:", round(cos(v[0], v[2]), 3))  # expect LOW

## Segment 3 — Build the RAG core

In [ ]:
# Cell 5 — chunking + vector store + retrieval
import faiss
from typing import List, Tuple

def chunk_text(text: str, size: int = 150, overlap: int = 30) -> List[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), size - overlap):
        chunks.append(" ".join(words[i:i + size]))
        if i + size >= len(words):
            break
    return chunks

class VectorStore:
    def __init__(self, embedder):
        self.embedder = embedder
        self.docs = []
        self.index = None

    def add(self, chunks: List[str]):
        self.docs.extend(chunks)
        emb = self.embedder.encode(self.docs)
        emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)   # cosine
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(emb.astype("float32"))

    def search(self, query: str, k: int = 3) -> List[Tuple[str, float]]:
        q = self.embedder.encode([query])
        q = q / np.linalg.norm(q, axis=1, keepdims=True)
        scores, ids = self.index.search(q.astype("float32"), k)
        # FAISS returns id -1 when k > number of chunks; skip those
        return [(self.docs[i], float(s)) for s, i in zip(scores[0], ids[0]) if 0 <= i < len(self.docs)]

In [ ]:
# Cell 6 — the RAG answer function (retrieve, then ask Groq)
def rag_answer(store, question, model="openai/gpt-oss-120b", k=3):
    hits = store.search(question, k)
    context = "\n\n".join(doc for doc, _ in hits)
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    r = client.chat.completions.create(model=model,
            messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content, hits

In [ ]:
# Cell 7 — quick test on a tiny sample (prove it works)
sample = """Groq is an inference engine that runs open-source LLMs very fast.
RAG means Retrieval-Augmented Generation: it fetches relevant chunks from
your own documents and gives them to the model as context, so the answer is
grounded in your data instead of the model's memory."""

store = VectorStore(embedder)
store.add(chunk_text(sample))
answer, hits = rag_answer(store, "What does RAG stand for?")
print(answer)

# PROBLEM: try a question the sample does NOT contain
print(rag_answer(store, "Who founded Groq?")[0])

## Segment 4 — Wrap it in a Streamlit app
`%%writefile` must be the first line. Everything below it is pure Python.

In [ ]:
%%writefile app.py
import os

import numpy as np
import streamlit as st
import faiss, PyPDF2, docx
from sentence_transformers import SentenceTransformer
from groq import Groq

st.set_page_config(page_title="RAG App with Groq", page_icon="📚")
st.title("📚 Open-Source RAG App with Groq")
st.caption("CS4106 — Foundations of Generative AI")

# ---- Settings (API key entered safely here, never hard-coded) ----
with st.sidebar:
    st.header("Settings")
    # Falls back to the GROQ_API_KEY env var / Space Secret if the box is left empty.
    api_key = st.text_input("Groq API key", type="password") or os.environ.get("GROQ_API_KEY", "")
    model = st.selectbox("Model", ["openai/gpt-oss-120b",
                                   "openai/gpt-oss-20b"])
    top_k = st.slider("Chunks to retrieve (k)", 1, 5, 3)


@st.cache_resource
def load_embedder():
    return SentenceTransformer("all-MiniLM-L6-v2")
embedder = load_embedder()


def read_file(f):
    name = f.name.lower()
    if name.endswith(".pdf"):
        return "".join(p.extract_text() or "" for p in PyPDF2.PdfReader(f).pages)
    if name.endswith(".docx"):
        return "\n".join(p.text for p in docx.Document(f).paragraphs)
    return f.read().decode("utf-8", "ignore")


def chunk_text(text, size=150, overlap=30):
    words = text.split(); out = []
    for i in range(0, len(words), size - overlap):
        out.append(" ".join(words[i:i + size]))
        if i + size >= len(words): break
    return out


def build_index(chunks):
    emb = embedder.encode(chunks)
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    idx = faiss.IndexFlatIP(emb.shape[1]); idx.add(emb.astype("float32"))
    return idx


def search(idx, chunks, query, k):
    q = embedder.encode([query]); q = q / np.linalg.norm(q, axis=1, keepdims=True)
    scores, ids = idx.search(q.astype("float32"), k)
    # FAISS pads with id -1 when k > number of chunks; skip those.
    return [(chunks[i], float(s)) for s, i in zip(scores[0], ids[0]) if 0 <= i < len(chunks)]


# ---- Upload & build the knowledge base ----
files = st.file_uploader("Upload PDF / DOCX / TXT",
                         type=["pdf", "docx", "txt"], accept_multiple_files=True)
if files and st.button("Build knowledge base"):
    text = "\n".join(read_file(f) for f in files)
    chunks = chunk_text(text)
    if not chunks:
        st.error("No text could be extracted. Scanned PDFs have no text layer — try a text-based file.")
    else:
        st.session_state.chunks = chunks
        st.session_state.index = build_index(chunks)
        st.success(f"Indexed {len(chunks)} chunks from {len(files)} file(s).")

# ---- Ask a question ----
question = st.text_input("Ask a question about your documents:")
if st.button("Answer") and question:
    if not api_key:
        st.error("Enter your Groq API key in the sidebar.")
    elif "index" not in st.session_state:
        st.error("Upload a document and click 'Build knowledge base' first.")
    else:
        hits = search(st.session_state.index, st.session_state.chunks, question, top_k)
        context = "\n\n".join(d for d, _ in hits)
        prompt = ("Answer using ONLY the context. If it is not there, say you "
                  f"don't know.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:")
        client = Groq(api_key=api_key.strip())
        r = client.chat.completions.create(model=model,
                messages=[{"role": "user", "content": prompt}])
        st.subheader("Answer")
        st.write(r.choices[0].message.content)
        with st.expander("Sources (retrieved chunks)"):
            for i, (d, s) in enumerate(hits, 1):
                st.markdown(f"**Chunk {i}** · score {s:.2f}")
                st.write(d)

## Segment 5 — Run it & open in your browser

In [ ]:
# Cell 9 — launch Streamlit (headless)
!pkill -9 -f streamlit 2>/dev/null
!streamlit run app.py --server.port 8501 --server.headless true &>/content/log.txt &
import time; time.sleep(15)   # give it time to start
!cat /content/log.txt          # look for: You can now view your Streamlit app

In [ ]:
# Cell 10 — open a public link (tunnel password = External URL IP from Cell 9's log)
!npx --yes localtunnel --port 8501